# Лабораторная 1. Языковая модель из счётчиков

Решение перенесено из `laba01` в структуру шаблона курса: `lab01/lab01.ipynb`.
Основная часть и бонусы 1 и 3 используют стандартную библиотеку и NumPy;
`torch` и `transformers` нужны только для бонуса 2.

Корпус: Ф. М. Достоевский, «Преступление и наказание»,
[Викитека](https://ru.wikisource.org/wiki/Преступление_и_наказание_(Достоевский)).
Корпуса лежат локально в `data/` и исключены из Git; подготовка описана в `data/README.md`.
Запуск: выбрать Python с зависимостями из корневого `requirements.txt` и выполнить ячейки сверху вниз.
Работают запуск из корня репозитория и запуск из `lab01/`.

По сравнению с прежним решением исправлены подписи температуры, повторное обучение,
учёт неизвестных слов и нормировка backoff. Прежние числа сохранены в `PREVIOUS_RESULTS.md`.

In [1]:
from pathlib import Path
from collections import Counter
import math
import re
import numpy as np

SEED = 42
np.random.seed(SEED)
LAB = Path.cwd() if (Path.cwd() / "data").is_dir() else Path.cwd() / "lab01"
DATA = LAB / "data"

def tokenize(text):
    return re.findall(r"[а-яёa-z]+", text.lower())

corpus_path = DATA / "corpus.txt"
if not corpus_path.exists():
    raise FileNotFoundError("Подготовьте lab01/data/corpus.txt по инструкции data/README.md")
tokens = tokenize(corpus_path.read_text(encoding="utf-8"))
assert len(tokens) >= 100_000
print(tokens[:30])
print("Всего слов:", len(tokens))
print("Словарь всего корпуса:", len(set(tokens)))

['в', 'начале', 'июля', 'в', 'чрезвычайно', 'жаркое', 'время', 'под', 'вечер', 'один', 'молодой', 'человек', 'вышел', 'из', 'своей', 'каморки', 'которую', 'нанимал', 'от', 'жильцов', 'в', 'с', 'м', 'переулке', 'на', 'улицу', 'и', 'медленно', 'как', 'бы']
Всего слов: 173006
Словарь всего корпуса: 24957


In [2]:
split_idx = int(len(tokens) * 0.9)

train_tokens = tokens[:split_idx]
test_tokens = tokens[split_idx:]

print("Обучающая часть:", len(train_tokens))
print("Тестовая часть:", len(test_tokens))

Обучающая часть: 155705
Тестовая часть: 17301


In [3]:
from collections import defaultdict

word_counts = defaultdict(int)
bigram_counts = defaultdict(lambda: defaultdict(int))

for v, w in zip(train_tokens[:-1], train_tokens[1:]):
    word_counts[v] += 1
    bigram_counts[v][w] += 1

In [4]:
word = "раскольников"

print("Количество:", word_counts[word])
print("Следующие слова:")
print(dict(bigram_counts[word]))

Количество: 520
Следующие слова:
{'студент': 1, 'немного': 1, 'и': 14, 'вышел': 4, 'тотчас': 4, 'не': 21, 'это': 3, 'а': 11, 'слушал': 2, 'успел': 2, 'что': 6, 'господи': 1, 'с': 14, 'в': 7, 'он': 5, 'ужасно': 1, 'бросился': 4, 'посмотрел': 1, 'вот': 3, 'говорил': 1, 'засмеялся': 4, 'оставшись': 2, 'быв': 1, 'отвернулся': 1, 'но': 8, 'преимущественно': 1, 'приходивший': 1, 'вдруг': 7, 'тут': 1, 'вздрогнул': 3, 'был': 4, 'потерялся': 1, 'сунул': 2, 'совсем': 1, 'сжимая': 1, 'на': 1, 'да': 9, 'стоял': 1, 'снял': 1, 'ступайте': 1, 'по': 3, 'тоже': 3, 'почувствовал': 2, 'уже': 7, 'обращаясь': 2, 'отдал': 1, 'поднял': 2, 'отвечал': 1, 'вырвал': 1, 'встал': 8, 'молча': 4, 'забыл': 1, 'затрепетал': 1, 'приподнялся': 2, 'задумчиво': 1, 'отстраняя': 1, 'у': 2, 'смотрел': 2, 'тем': 3, 'молчал': 6, 'смотря': 1, 'взглянул': 1, 'быстро': 3, 'тревожно': 1, 'задумался': 1, 'отмахиваясь': 1, 'после': 2, 'приподнявшись': 1, 'едва': 2, 'оборотился': 1, 'мутным': 1, 'опускаясь': 1, 'господин': 1, 'промям

In [5]:
sorted(
    bigram_counts["раскольников"].items(),
    key=lambda x: x[1],
    reverse=True
)[:10]

[('не', 21),
 ('и', 14),
 ('с', 14),
 ('я', 12),
 ('а', 11),
 ('да', 9),
 ('но', 8),
 ('встал', 8),
 ('как', 8),
 ('в', 7)]


## 1. Корпус и статистика

Слова приводятся к нижнему регистру и выделяются выражением `[а-яёa-z]+`.
Пунктуация и числа удалены ради простой модели слов; дефис разделяет токены.
Стоп-слова сохраняются, иначе изменилось бы соседство слов.
Последние 10 % откладываются до построения модели, словарь строится только по обучению.

In [6]:
vocabulary = set(train_tokens)
top_20 = Counter(train_tokens).most_common(20)
print(f"Всего слов: {len(tokens)}")
print(f"Размер обучающей части: {len(train_tokens)}")
print(f"Размер тестовой части: {len(test_tokens)}")
print(f"Размер словаря обучения без <unk>: {len(vocabulary)}")
print("\n20 самых частых слов обучения:")
for i, (word, count) in enumerate(top_20, start=1):
    print(f"{i:2}. {word:<15} {count}")

Всего слов: 173006
Размер обучающей части: 155705
Размер тестовой части: 17301
Размер словаря обучения без <unk>: 23156

20 самых частых слов обучения:
 1. и               7593
 2. не              3398
 3. в               3319
 4. что             3114
 5. он              2496
 6. с               2374
 7. я               2207
 8. на              2159
 9. то              1954
10. а               1689
11. как             1639
12. это             1293
13. все             1199
14. так             1089
15. же              1034
16. да              1028
17. его             1024
18. но              1002
19. вы              943
20. к               820


## 2. Счётчики и вероятности

`word_counts` содержит частоты всех слов, `context_counts` — число исходящих переходов
(у последнего токена нет продолжения). Словарь включает `<unk>`: все неизвестные слова
проверки отображаются в один токен. Это не добавляет тестовые слова в обучение.
При α > 0 сумма вероятностей по этому фиксированному словарю равна 1 даже для неизвестного контекста.
При α = 0 ненаблюдавшийся переход имеет вероятность 0.

In [7]:
from collections import Counter, defaultdict

class BigramLanguageModel:
    UNK = "<unk>"

    def fit(self, tokens):
        self.vocabulary = set(tokens) | {self.UNK}
        self.word_counts = Counter(tokens)
        self.context_counts = Counter()
        self.bigram_counts = defaultdict(Counter)
        for v, w in zip(tokens, tokens[1:]):
            self.context_counts[v] += 1
            self.bigram_counts[v][w] += 1
        return self

    def known(self, word):
        return word if word in self.vocabulary else self.UNK

    def prob(self, v, w, alpha=0):
        if alpha < 0:
            raise ValueError("alpha должна быть неотрицательной")
        v, w = self.known(v), self.known(w)
        denominator = self.context_counts[v] + alpha * len(self.vocabulary)
        if denominator == 0:
            return 0.0
        return (self.bigram_counts.get(v, {}).get(w, 0) + alpha) / denominator

    def generate(self, start_word, length=30, temperature=1.0):
        if temperature <= 0 or length < 1:
            raise ValueError("Температура и длина должны быть положительными")
        result = [start_word]
        for _ in range(length - 1):
            row = self.bigram_counts.get(result[-1], {})
            if not row:
                break
            words = list(row)
            weights = np.array([row[word] for word in words], dtype=float)
            weights = (weights / weights.max()) ** (1 / temperature)
            result.append(str(np.random.choice(words, p=weights / weights.sum())))
        return " ".join(result)

    def logprob(self, text, alpha=0):
        words = tokenize(text)
        probabilities = [self.prob(v, w, alpha) for v, w in zip(words, words[1:])]
        return sum(math.log(p) for p in probabilities) if all(probabilities) else -math.inf

    def perplexity(self, tokens, alpha=1.0):
        if len(tokens) < 2:
            raise ValueError("Для перплексии нужны хотя бы два токена")
        probabilities = [self.prob(v, w, alpha) for v, w in zip(tokens, tokens[1:])]
        if not all(probabilities):
            return math.inf
        return math.exp(-sum(math.log(p) for p in probabilities) / (len(tokens) - 1))

In [8]:
model = BigramLanguageModel()

model.fit(train_tokens)

In [9]:
# Вероятность слова человек, после слова молодой
model.prob("молодой", "человек")

0.7954545454545454


In [10]:
#Проверка, что сумма вероятностей переходов от слова к остальным другим равна 1
word = "раскольников"

total_prob = 0

for next_word in model.bigram_counts[word]:
    total_prob += model.prob(word, next_word)

print(total_prob)

0.9999999999999978


Пункт 3. Генерация. Пять текстов по тридцать слов от разных стартовых слов. Каждое следующее слово выбирается случайно по вероятностям из строки предыдущего.

In [11]:
start_words = [
    "лошадь",
    "соня",
    "порфирий",
    "человек",
    "он",
    "ниссан"
]

for word in start_words:
    print(f"\nСтартовое слово: {word}")
    print(model.generate(word, 30))


Стартовое слово: лошадь
лошадь потому по женскому отделению такие мужики кнутами иной раз стиснул зубы и выкупила меня точных объяснений в памяти вот ведь уж не так как я действительно надо снять с

Стартовое слово: соня
соня свою руку и в заключении речи то крикнул хозяин очутившийся опять таки по улице на меня это только вдруг самый неожиданный результат рассчитывать на седьмую версту милости просим я

Стартовое слово: порфирий
порфирий петрович как он на соню и даже меня и никаких то скучно развлекись время под предлогом похорон супруги отправился к тебе все доставал а уж очень даже может больше

Стартовое слово: человек
человек а попала в комнату смежную с концом так чего он пошел ибо зачем я совершенно забыл спросить эх маху я боюсь я целый час когда андрей семенович как пион

Стартовое слово: он
он ей человек обязан например на воротах вам не знаю знаю все надеется что все переносила и что не петрушку же тебе прямо под которым все более насущным заботам и

Стартовое слово: нисса

## 4. Температура

Используем $p_i^{1/T}$ с перенормировкой: при T=0.5 усиливаются частые продолжения,
при T=1 сохраняется исходное распределение, при T=2 чаще выбираются редкие слова.

In [12]:
for temperature in [0.5, 1, 2]:
    print(f"T = {temperature}")
    print(model.generate("раскольников", 30, temperature=temperature))
    print()

T = 0.5
раскольников и что я так и уж не видал и внушительно заметила через четверть часа вдруг в этом роде но не был в это я тогда в то есть не

T = 1
раскольников я он слушал приходил не в неделю нечаянно вошедшую во первых потому труд так вы как бы стал беспокоить хотя и ей вдруг поднял я ненавижу подле него там

T = 2
раскольников говорите или с продолжал соображать ну так бедно одета как все наши новости реформы идеи как волнует например слишком уже сбросил с толковым взглядом раскольников стоявший в противном же



## 5. Правдоподобие и сглаживание

Используем тот же связный фрагмент и ту же перестановку, что в исходном эксперименте с Qwen,
чтобы сохранить сопоставимость. Перестановка зафиксирована, а не выбирается заново в каждой ячейке.

In [13]:
test_sentence = " ".join(test_tokens[100:115])

shuffled_sentence = "его уж ее скорее отпустит он конечно убьет она в и умрет чем теперь и"

print("Связный текст:")
print(test_sentence)
print("logprob:", model.logprob(test_sentence))

print("\nПеремешанный текст:")
print(shuffled_sentence)
print("logprob:", model.logprob(shuffled_sentence))

Связный текст:
он скорее умрет чем отпустит ее и и уж конечно она убьет его теперь в
logprob: -inf

Перемешанный текст:
его уж ее скорее отпустит он конечно убьет она в и умрет чем теперь и
logprob: -inf


Без сглаживания корректно сравнивать правдоподобие текстов не всегда возможно.
Если хотя бы одна биграмма не встречалась в обучающей части корпуса,
ее вероятность равна нулю. Тогда логарифм вероятности равен
$-\infty$, и вся сумма log-probability также становится $-\infty$.

In [14]:
test_sentence = " ".join(test_tokens[100:115])

shuffled_sentence = "его уж ее скорее отпустит он конечно убьет она в и умрет чем теперь и"

print("Связный текст:")
print(test_sentence)

print("\nБез сглаживания:")
print(model.logprob(test_sentence))

print("\nСо сглаживанием:")
print(model.logprob(test_sentence, alpha=1))

Связный текст:
он скорее умрет чем отпустит ее и и уж конечно она убьет его теперь в

Без сглаживания:
-inf

Со сглаживанием:
-122.07123918709972


In [15]:
print("Перемешанный текст:")
print(shuffled_sentence)

print("\nБез сглаживания:")
print(model.logprob(shuffled_sentence))

print("\nСо сглаживанием:")
print(model.logprob(shuffled_sentence, alpha=1))

Перемешанный текст:
его уж ее скорее отпустит он конечно убьет она в и умрет чем теперь и

Без сглаживания:
-inf

Со сглаживанием:
-132.7764016362663


Сглаживание Лапласа добавляет небольшую ненулевую вероятность
невиданным переходам. После этого можно сравнивать log-probability:
чем больше значение (то есть чем оно ближе к нулю), тем более
правдоподобным модель считает текст.

Пункт 6. Перплексия

In [16]:
alphas = [1, 0.1, 0.01]

for alpha in alphas:
    ppl = model.perplexity(test_tokens, alpha=alpha)

    print(f"alpha = {alpha}: perplexity = {ppl:.2f}")

alpha = 1: perplexity = 10024.66
alpha = 0.1: perplexity = 5373.19
alpha = 0.01: perplexity = 4118.69


Перплексия была рассчитана на отложенных 10 % корпуса, которые
не использовались при обучении модели.

Меньшее значение perplexity соответствует лучшему качеству модели,
так как означает более высокую вероятность реальной
последовательности слов.

Слишком большое значение параметра сглаживания ухудшает результат,
поскольку перераспределяет слишком большую долю вероятности
на невиданные биграммы и уменьшает вероятность реально
наблюдавшихся переходов.

In [17]:
results = []

for alpha in [1, 0.1, 0.01]:
    ppl = model.perplexity(test_tokens, alpha)
    results.append((alpha, ppl))

print("alpha\tperplexity")

for alpha, ppl in results:
    print(f"{alpha}\t{ppl:.2f}")

alpha	perplexity
1	10024.66
0.1	5373.19
0.01	4118.69


In [18]:
alphas_extended = [
    1,
    0.5,
    0.1,
    0.05,
    0.01,
    0.005,
    0.001
]

results_extended = []

for alpha in alphas_extended:
    ppl = model.perplexity(test_tokens, alpha=alpha)
    results_extended.append((alpha, ppl))

print("alpha\tperplexity")

for alpha, ppl in results_extended:
    print(f"{alpha:<6}\t{ppl:.2f}")

alpha	perplexity
1     	10024.66
0.5   	8208.55
0.1   	5373.19
0.05  	4708.33
0.01  	4118.69
0.005 	4209.08
0.001 	5285.07


## Бонус 1. Триграммы

В старой версии наблюдавшиеся триграммы уже получали суммарную вероятность 1,
а к ним добавлялись вероятности невиданных продолжений из биграмм. Поэтому сумма превышала 1,
и прежние 3395.89 нельзя считать корректной перплексией.

Теперь из частоты каждого наблюдавшегося продолжения вычитается discount=0.5.
Освободившаяся масса распределяется между невиданными продолжениями пропорционально
биграммным вероятностям. Для невиданного контекста используется полная биграммная строка.
При оценке обе модели предсказывают одни и те же N−1 переходов.
Генерация, как и раньше, выбирает среди наблюдавшихся продолжений; если контекста нет, откатывается на биграммы.

In [19]:
class TrigramLanguageModel:
    """Абсолютное дисконтирование и нормированный откат на биграммы."""
    def __init__(self, bigram_model, discount=0.5):
        if not 0 < discount < 1:
            raise ValueError("discount должен лежать между 0 и 1")
        self.bigram_model = bigram_model
        self.discount = discount

    def fit(self, tokens):
        self.context_counts = Counter()
        self.trigram_counts = defaultdict(Counter)
        self._backoff_cache = {}
        for w1, w2, w3 in zip(tokens, tokens[1:], tokens[2:]):
            self.context_counts[w1, w2] += 1
            self.trigram_counts[w1, w2][w3] += 1
        return self

    def prob(self, w1, w2, w3, alpha=0.01):
        if alpha <= 0:
            raise ValueError("Для backoff требуется alpha > 0")
        w1, w2, w3 = map(self.bigram_model.known, (w1, w2, w3))
        context = (w1, w2)
        row = self.trigram_counts.get(context, {})
        if not row:
            return self.bigram_model.prob(w2, w3, alpha)
        count = self.context_counts[context]
        if w3 in row:
            return (row[w3] - self.discount) / count
        key = (context, alpha)
        if key not in self._backoff_cache:
            leftover = self.discount * len(row) / count
            unseen_mass = 1 - sum(self.bigram_model.prob(w2, w, alpha) for w in row)
            self._backoff_cache[key] = leftover / unseen_mass
        return self._backoff_cache[key] * self.bigram_model.prob(w2, w3, alpha)

    def generate(self, start_words, length=30, temperature=1.0):
        if temperature <= 0 or length < 2:
            raise ValueError("Нужны temperature > 0 и length >= 2")
        result = list(start_words)
        for _ in range(length - 2):
            row = self.trigram_counts.get(tuple(result[-2:]), {})
            if not row:
                row = self.bigram_model.bigram_counts.get(result[-1], {})
            if not row:
                break
            words = list(row)
            weights = np.array([row[w] for w in words], dtype=float)
            weights = (weights / weights.max()) ** (1 / temperature)
            result.append(str(np.random.choice(words, p=weights / weights.sum())))
        return " ".join(result)

    def perplexity(self, tokens, alpha=0.01):
        if len(tokens) < 3:
            raise ValueError("Нужны хотя бы три токена")
        # Оцениваем те же N-1 переходов, что и биграммная модель.
        total = math.log(self.bigram_model.prob(tokens[0], tokens[1], alpha))
        total += sum(math.log(self.prob(a, b, c, alpha))
                     for a, b, c in zip(tokens, tokens[1:], tokens[2:]))
        return math.exp(-total / (len(tokens) - 1))

In [20]:
trigram_model = TrigramLanguageModel(model)

trigram_model.fit(train_tokens)

In [21]:
context = ("молодой", "человек")

print("Контекст встретился:", trigram_model.context_counts[context], "раз")
print("Продолжения:")
print(dict(trigram_model.trigram_counts[context]))

Контекст встретился: 35 раз
Продолжения:
{'вышел': 1, 'проходя': 1, 'вдруг': 1, 'был': 1, 'переступил': 1, 'взглянул': 1, 'с': 2, 'ни': 1, 'ваша': 1, 'взял': 1, 'оставшись': 1, 'спорить': 1, 'несколько': 1, 'отчасти': 1, 'случалось': 1, 'можете': 1, 'не': 2, 'подтвердил': 1, 'продолжал': 1, 'видел': 1, 'поспешил': 1, 'конечно': 1, 'смотрите': 1, 'и': 2, 'лет': 1, 'что': 1, 'имеющий': 1, 'как': 1, 'очень': 1, 'воскликнула': 1, 'свидригайлов': 1, 'может': 1}


In [22]:
start_pairs = [
    ("молодой", "человек"),
    ("родион", "романович"),
    ("он", "не"),
    ("в", "эту"),
    ("и", "вдруг"),
    ("ниссан", "скайлан")
]

for pair in start_pairs:
    print(f"\nСтарт: {pair}")
    print(
        trigram_model.generate(
            pair,
            length=30
        )
    )


Старт: ('молодой', 'человек')
молодой человек продолжал он с упоением помышлял в глубочайшем секрете о девице благонравной и бедной непременно бедной очень молоденькой очень хорошенькой благородной и образованной очень запуганной чрезвычайно много испытавшей несчастий

Старт: ('родион', 'романович')
родион романович уж извините меня старика человек еще молодой с толстою такою шеей и с грустною улыбкой посмотрел на него сегодня начал зосимов очень обрадовавшись пришедшим потому что его хотят

Старт: ('он', 'не')
он не поднял а это больней с больней с когда помутилось сердце человеческое когда цитуется фраза что кровь освежает когда вся опасность прошла я и сам еще и разъяснял разумихину

Старт: ('в', 'эту')
в эту минуту не будет а только вид показываете странно ответил незнакомый спокойно рассмеявшись аркадий иванович право так и покатилась со смеху за ним с совершенно веселым лукавым и нисколько

Старт: ('и', 'вдруг')
и вдруг как громом да и вы дрожите порфирий петрович впрочем все эт

In [23]:
alpha = 0.01

bigram_ppl = model.perplexity(
    test_tokens,
    alpha=alpha
)

trigram_ppl = trigram_model.perplexity(
    test_tokens,
    alpha=alpha
)

print(f"Bigram perplexity:  {bigram_ppl:.2f}")
print(f"Trigram perplexity: {trigram_ppl:.2f}")

Bigram perplexity:  4118.69
Trigram perplexity: 4428.28


In [24]:
print("BIGRAM:")
print(
    model.generate(
        "молодой",
        length=30,
        temperature=1
    )
)

BIGRAM:
молодой человек никогда и когда тут и хмурые на него такой развязке а вразумихин студент разболтался и ушел вот дверь и более подкипятило злобу мою сестру но нарочно даже и


In [25]:
print("\nTRIGRAM:")
print(
    trigram_model.generate(
        ("молодой", "человек"),
        length=30,
        temperature=1
    )
)


TRIGRAM:
молодой человек очень уж вы теперь будьте покойны она его заключила пульхерия александровна вышла из себя выходя на лестницу надо вверх головой вносить оборачивайте вот так я просто был каждую


### Результат исправленного backoff

Текущие значения напечатаны выше. Увеличение контекста не гарантирует улучшения:
триграммы разрежены, а результат зависит от дисконтирования и сглаживания.
Параметры на тесте не оптимизировались. Старые результаты сохранены отдельно.

## Бонус 2. Сравнение с Qwen3-0.6B-Base

Эта часть требует `torch`, `transformers` и весов модели (скачиваются при первом запуске).
У нас веса уже сохранены в локальном кэше. Используется та же пара текстов, что и в основной части.

In [26]:
import torch
import math

from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen3-0.6B-Base"

device = "cuda" if torch.cuda.is_available() else (
    "mps" if torch.backends.mps.is_available() else "cpu"
)

print("device:", device)

qwen_tok = AutoTokenizer.from_pretrained(MODEL_ID)

qwen_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32
)

qwen_model = qwen_model.to(device).eval()

device: cuda


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|##########| 310/310 [00:00<00:00, 363.71it/s]


In [27]:
@torch.no_grad()
def qwen_log_likelihood(text):
    ids = qwen_tok(
        text,
        return_tensors="pt"
    ).to(device)["input_ids"]

    logits = qwen_model(ids).logits[0, :-1].float()

    logp = torch.log_softmax(
        logits,
        dim=-1
    )

    target = ids[0, 1:]

    token_logp = logp[
        torch.arange(target.numel(), device=device),
        target
    ].cpu()

    return token_logp.sum().item(), token_logp.numel()

In [28]:
qwen_connected, n_connected = qwen_log_likelihood(
    test_sentence
)

qwen_shuffled, n_shuffled = qwen_log_likelihood(
    shuffled_sentence
)

print("Связный текст:")
print(test_sentence)
print(f"log P = {qwen_connected:.2f}")

print("\nПеремешанный текст:")
print(shuffled_sentence)
print(f"log P = {qwen_shuffled:.2f}")

Связный текст:
он скорее умрет чем отпустит ее и и уж конечно она убьет его теперь в
log P = -101.33

Перемешанный текст:
его уж ее скорее отпустит он конечно убьет она в и умрет чем теперь и
log P = -117.66


In [29]:
bigram_connected = model.logprob(
    test_sentence,
    alpha=1
)

bigram_shuffled = model.logprob(
    shuffled_sentence,
    alpha=1
)

bigram_gap = bigram_connected - bigram_shuffled
qwen_gap = qwen_connected - qwen_shuffled

print("Разница между связным и перемешанным текстом:")
print(f"Bigram: {bigram_gap:.2f}")
print(f"Qwen:   {qwen_gap:.2f}")

Разница между связным и перемешанным текстом:
Bigram: 10.71
Qwen:   16.32


### Интерпретация сравнения

Обе модели предпочитают исходный порядок слов; значения и разрывы напечатаны выше.
Биграммная модель работает со словами, Qwen — с субсловными токенами, поэтому абсолютные
log P и величины разрывов между моделями не являются сопоставимой метрикой качества.
Одна пара текстов иллюстрирует эффект порядка слов, но не доказывает общее превосходство модели.

## Бонус 3. Наивный Байес: определение автора

Используем отдельно подготовленные EPUB-корпуса:
Достоевский — «Преступление и наказание» (173 138 токенов),
Толстой — «Анна Каренина» (271 303 токена).
Они отличаются по подготовке от корпуса основной части; числа и разбиения не смешиваются.
Последние 10 % каждой книги проверяются непересекающимися фрагментами по 30 слов.
Это классификация двух конкретных книг: имена персонажей и тематика могут влиять на результат,
поэтому accuracy не означает такую же точность на других произведениях этих авторов.

In [30]:
dostoevsky_tokens = (DATA / "author_dostoevsky_tokens.txt").read_text(encoding="utf-8").split()
tolstoy_tokens = (DATA / "author_tolstoy_tokens.txt").read_text(encoding="utf-8").split()
print("Достоевский:", len(dostoevsky_tokens))
print("Толстой:", len(tolstoy_tokens))

Достоевский: 173138
Толстой: 271303


In [31]:
def train_test_split(tokens, train_ratio=0.9):
    split_idx = int(len(tokens) * train_ratio)

    train = tokens[:split_idx]
    test = tokens[split_idx:]

    return train, test

In [32]:
dost_train, dost_test = train_test_split(dostoevsky_tokens)
tolstoy_train, tolstoy_test = train_test_split(tolstoy_tokens)

print("Достоевский:")
print("train:", len(dost_train))
print("test:", len(dost_test))

print("\nТолстой:")
print("train:", len(tolstoy_train))
print("test:", len(tolstoy_test))

Достоевский:
train: 155824
test: 17314

Толстой:
train: 244172
test: 27131


In [33]:
from collections import defaultdict
import numpy as np


class NaiveBayesAuthorClassifier:
    def __init__(self, alpha=1.0):
        self.alpha = alpha

        self.word_counts = {
            "dostoevsky": defaultdict(int),
            "tolstoy": defaultdict(int)
        }

        self.total_words = {
            "dostoevsky": 0,
            "tolstoy": 0
        }

        self.vocabulary = set()

    def fit(self, dost_tokens, tolstoy_tokens):
        for word in dost_tokens:
            self.word_counts["dostoevsky"][word] += 1
            self.total_words["dostoevsky"] += 1
            self.vocabulary.add(word)

        for word in tolstoy_tokens:
            self.word_counts["tolstoy"][word] += 1
            self.total_words["tolstoy"] += 1
            self.vocabulary.add(word)

    def word_prob(self, word, author):
        V = len(self.vocabulary)

        return (
            self.word_counts[author][word] + self.alpha
        ) / (
            self.total_words[author] + self.alpha * V
        )

    def score(self, tokens, author):
        total_logprob = 0.0

        for word in tokens:
            p = self.word_prob(word, author)
            total_logprob += np.log(p)

        return total_logprob

    def predict(self, tokens):
        dost_score = self.score(
            tokens,
            "dostoevsky"
        )

        tolstoy_score = self.score(
            tokens,
            "tolstoy"
        )

        if dost_score > tolstoy_score:
            return "Достоевский"

        return "Толстой"

In [34]:
author_model = NaiveBayesAuthorClassifier(
    alpha=1.0
)

author_model.fit(
    dost_train,
    tolstoy_train
)

In [35]:
fragment = dost_test[100:130]

print(" ".join(fragment))

print(
    author_model.predict(fragment)
)

решимостью воспаленно страстным тяжелым взглядом дуня поняла что он скорее умрет чем отпустит ее и и уж конечно она убьет его теперь в двух шагах вдруг она отбросила револьвер бросила
Достоевский


In [36]:
fragment = tolstoy_test[100:130]

print(" ".join(fragment))

print(
    author_model.predict(fragment)
)

не мог всего обнять слушая графиню лидию ивановну и чувствуя устремленные на себя красивые наивные или плутовские он сам не знал глаза landau степан аркадьич начинал испытывать какую то особенную
Толстой


In [37]:
def make_fragments(tokens, fragment_size=30):
    fragments = []

    for i in range(0, len(tokens) - fragment_size + 1, fragment_size):
        fragment = tokens[i:i + fragment_size]
        fragments.append(fragment)

    return fragments

In [38]:
dost_fragments = make_fragments(
    dost_test,
    fragment_size=30
)

tolstoy_fragments = make_fragments(
    tolstoy_test,
    fragment_size=30
)

print("Фрагментов Достоевского:", len(dost_fragments))
print("Фрагментов Толстого:", len(tolstoy_fragments))

Фрагментов Достоевского: 577
Фрагментов Толстого: 904


In [39]:
correct = 0
total = 0

for fragment in dost_fragments:
    prediction = author_model.predict(fragment)

    if prediction == "Достоевский":
        correct += 1

    total += 1


for fragment in tolstoy_fragments:
    prediction = author_model.predict(fragment)

    if prediction == "Толстой":
        correct += 1

    total += 1


accuracy = correct / total

print("Правильно:", correct)
print("Всего:", total)
print(f"Accuracy: {accuracy:.4f}")
print(f"Accuracy: {accuracy * 100:.2f}%")

Правильно: 1350
Всего: 1481
Accuracy: 0.9115
Accuracy: 91.15%


In [40]:
dost_correct = 0

for fragment in dost_fragments:
    if author_model.predict(fragment) == "Достоевский":
        dost_correct += 1


tolstoy_correct = 0

for fragment in tolstoy_fragments:
    if author_model.predict(fragment) == "Толстой":
        tolstoy_correct += 1


dost_accuracy = dost_correct / len(dost_fragments)
tolstoy_accuracy = tolstoy_correct / len(tolstoy_fragments)

print(
    f"Достоевский: {dost_accuracy * 100:.2f}%"
)

print(
    f"Толстой: {tolstoy_accuracy * 100:.2f}%"
)

Достоевский: 86.66%
Толстой: 94.03%


In [41]:
confusion = {
    "Достоевский": {
        "Достоевский": 0,
        "Толстой": 0
    },
    "Толстой": {
        "Достоевский": 0,
        "Толстой": 0
    }
}

for fragment in dost_fragments:
    pred = author_model.predict(fragment)

    confusion["Достоевский"][pred] += 1


for fragment in tolstoy_fragments:
    pred = author_model.predict(fragment)

    confusion["Толстой"][pred] += 1


print("Настоящий \\ Предсказанный")
print()

print(
    "Достоевский:",
    confusion["Достоевский"]
)

print(
    "Толстой:",
    confusion["Толстой"]
)

Настоящий \ Предсказанный

Достоевский: {'Достоевский': 500, 'Толстой': 77}
Толстой: {'Достоевский': 54, 'Толстой': 850}


## Выводы

Биграммы воспроизводят локальные сочетания, но теряют сюжет и дальнее согласование.
Сглаживание делает оценки невиданных пар конечными; слишком большое α сглаживает полезные различия,
слишком маленькое сильно штрафует редкие переходы. Из проверенных α наименьшую перплексию даёт 0.01;
это описание эксперимента на отложенной части, а не независимая оценка после подбора параметров.
Бонусы показывают расширение счётчиков на больший контекст и классификацию книг.